# 04. 노선 단위 분석

**질문**: GTX와 목적지가 겹치는 노선(연신내역·서울역 방면 하차가 많은 노선)이 더 많이 줄었나?

- 단위: 노선 24개 (셀보다 승차 규모가 커서 잡음이 작다)
- 경쟁 목적지 비율: 2024년 승차 중 연신내역·서울역 1km 이내 하차 비율
- 셀이 아닌 **전체 원자료** 기준이다 (공급구조 유지 셀로 제한하지 않음)

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # 정리본 최상위의 config.py

from config import *
setup_notebook()

import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats

data = load_model_data()  # data/processed/09_회귀분석_격자통합데이터.csv

In [2]:
trips = []
for year, path in RAW_PATHS.items():
    raw = read_csv(path)
    hour = pd.to_datetime(raw["ride_dt"].astype(str), format="%Y%m%d%H%M%S", errors="coerce").dt.hour
    trips.append(pd.DataFrame({
        "노선": raw["query_route_no"].astype(str), "연도": year,
        "승차인원": pd.to_numeric(raw["utztn_nope"], errors="coerce").fillna(0),
        "오전": hour.between(*MORNING_HOURS),
        "경쟁목적지": is_competing_destination(raw["하차위도"], raw["하차경도"]),
        "GTX-A 영향역": raw["GTX_A_역"],
    }))
trips = pd.concat(trips, ignore_index=True)

all_day = trips.pivot_table(index="노선", columns="연도", values="승차인원", aggfunc="sum", fill_value=0)
morning = trips[trips.오전].pivot_table(index="노선", columns="연도", values="승차인원", aggfunc="sum", fill_value=0)
base = trips[trips.연도 == 2024]
route = pd.DataFrame({
    "GTX-A 영향역": base.groupby("노선")["GTX-A 영향역"].first(),
    "2024 종일": all_day[2024], "2025 종일": all_day[2025],
    "2024 오전": morning[2024], "2025 오전": morning[2025],
    "경쟁목적지 비율(%)": base[base.경쟁목적지].groupby("노선").승차인원.sum() / base.groupby("노선").승차인원.sum() * 100,
}).fillna({"경쟁목적지 비율(%)": 0})
route["종일 변화율(%)"] = (route["2025 종일"] / route["2024 종일"] - 1) * 100
route["오전 변화율(%)"] = (route["2025 오전"] / route["2024 오전"] - 1) * 100
route["노선 유형"] = np.select([route.index.str.startswith("M"), route.index.str.startswith("G")], ["M버스", "G버스"], "일반 광역")
route.to_csv(ROUTE_LEVEL_PATH, encoding="utf-8-sig")
route.sort_values("종일 변화율(%)").round(1)

,GTX-A 영향역,2024 종일,2025 종일,2024 오전,2025 오전,경쟁목적지 비율(%),종일 변화율(%),오전 변화율(%),노선 유형
노선,,,,,,,,,
M7111,운정중앙역,1319,491,431,191,26.2,-62.8,-55.7,M버스
M7154,운정중앙역,388,168,147,65,55.7,-56.7,-55.8,M버스
9700,대곡역·킨텍스역,224,113,84,38,0.0,-49.6,-54.8,일반 광역
1100,킨텍스역,99,54,61,11,20.2,-45.5,-82.0,일반 광역
G7625,운정중앙역,709,444,248,119,0.0,-37.4,-52.0,G버스
7101,운정중앙역,599,401,254,169,0.0,-33.1,-33.5,일반 광역
9600,대곡역,145,103,37,27,0.0,-29.0,-27.0,일반 광역
G7426,운정중앙역,941,672,220,195,0.0,-28.6,-11.4,G버스
1200,대곡역·킨텍스역,2629,1907,1070,735,23.5,-27.5,-31.3,일반 광역


In [3]:
for label, column in [("종일", "종일 변화율(%)"), ("오전", "오전 변화율(%)")]:
    rho, p = stats.spearmanr(route["경쟁목적지 비율(%)"], route[column])
    print(f"Spearman(경쟁목적지 비율, {label} 변화율) = {rho:.3f}, p = {p:.3f}")
route_model = sm.GLM(route["2025 종일"], sm.add_constant(route[["경쟁목적지 비율(%)"]]),
                     family=sm.families.Poisson(), offset=np.log(route["2024 종일"])).fit(cov_type="HC1")
display(coef_table(route_model, rate=True))

summary = {}
for column in ["노선 유형", "GTX-A 영향역"]:
    g = route.groupby(column)[["2024 종일", "2025 종일"]].sum()
    g["변화율(%)"] = (g["2025 종일"] / g["2024 종일"] - 1) * 100
    summary[column] = g.round(1)
display(summary["노선 유형"])
summary["GTX-A 영향역"].sort_values("변화율(%)")

Spearman(경쟁목적지 비율, 종일 변화율) = -0.001, p = 0.996
Spearman(경쟁목적지 비율, 오전 변화율) = -0.033, p = 0.878


,coef,p,변화율 차이(%),유의
const,-0.1539,0.0176,-14.2680,**
경쟁목적지 비율(%),-0.0024,0.4603,-0.2412,


,2024 종일,2025 종일,변화율(%)
노선 유형,,,
G버스,1650,1116,-32.4
M버스,6199,5666,-8.6
일반 광역,22484,18470,-17.9


,2024 종일,2025 종일,변화율(%)
GTX-A 영향역,,,
운정중앙역,9191,6458,-29.7
운정중앙역·킨텍스역,3385,2572,-24.0
대곡역·킨텍스역,10780,8903,-17.4
대곡역,1454,1214,-16.5
대곡역·운정중앙역,916,942,2.8
킨텍스역,4607,5163,12.1


## 결론
- 경쟁 목적지 비율과 노선 감소율 사이에는 **관계가 없다**(Spearman ≈ 0).
- 크게 줄어든 노선은 **M7111(−63%), M7154(−57%)** 등 운정중앙역 권역 노선이다. 반대로 M7731·M7412는 늘었다.
- 감소를 설명하는 것은 **'어디로 가느냐(목적지)'가 아니라 '어디서 타느냐(GTX역 접근성)'**다. 셀 단위 결과(03)와 일관된다.
- 참고: 경쟁 목적지를 연신내·서울역 1km로 정의한 것이 너무 좁을 수 있다. GTX-A 정차역이나 환승 목적지까지 넓히는 것을 검토할 만하다.